In [1]:
import sys
sys.path.append(r"C:\Users\prash\toposegnet\src")

import os
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import DataLoader
from tqdm import tqdm
import segmentation_models_pytorch as smp
import json

from dataset import CloudSEN12Dataset

# ── config ───────────────────────────────────────────────────────────────────
EXPERIMENTS = [
    {
        "name"         : "1b — no terrain, augment=True",
        "checkpoint"   : r"C:\Users\prash\toposegnet\experiments\unet_none_augTrue\best_model.pth",
        "terrain_mode" : "none",
        "in_channels"  : 13,
    },
    {
        "name"         : "1a — no terrain, augment=False",
        "checkpoint"   : r"C:\Users\prash\toposegnet\experiments\unet_none_augFalse\best_model.pth",
        "terrain_mode" : "none",
        "in_channels"  : 13,
    },
]

# ── metrics ──────────────────────────────────────────────────────────────────
def compute_metrics(preds, labels, num_classes=4):
    iou_per_class = []
    f1_per_class  = []

    for c in range(num_classes):
        pred_c  = (preds  == c)
        label_c = (labels == c)

        tp = (pred_c & label_c).sum().item()
        fp = (pred_c & ~label_c).sum().item()
        fn = (~pred_c & label_c).sum().item()

        iou = tp / (tp + fp + fn + 1e-8)
        f1  = 2*tp / (2*tp + fp + fn + 1e-8)

        iou_per_class.append(iou)
        f1_per_class.append(f1)

    return {
        "iou"  : iou_per_class,
        "miou" : np.mean(iou_per_class),
        "f1"   : f1_per_class,
        "mf1"  : np.mean(f1_per_class),
    }

# ── evaluate ─────────────────────────────────────────────────────────────────
@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    all_preds  = []
    all_labels = []

    for images, labels in tqdm(loader, desc="  evaluating", leave=False):
        images = images.to(device)
        labels = labels.to(device).long()

        logits = model(images)
        preds  = logits.argmax(dim=1)

        all_preds.append(preds.cpu())
        all_labels.append(labels.cpu())

    all_preds  = torch.cat(all_preds,  dim=0)
    all_labels = torch.cat(all_labels, dim=0)

    return compute_metrics(all_preds, all_labels)

# ── main ─────────────────────────────────────────────────────────────────────
def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}\n")

    # test dataloader — same for all experiments
    test_ds = CloudSEN12Dataset(
        split="Test",
        terrain_mode="none",
        augment=False
    )
    test_loader = DataLoader(
        test_ds,
        batch_size=2,
        shuffle=False,
        num_workers=0,
        pin_memory=True,
    )

    results = {}

    for exp in EXPERIMENTS:
        print(f"=== {exp['name']} ===")

        model = smp.Unet(
            encoder_name   = "resnet34",
            encoder_weights= None,          # loading our own weights
            in_channels    = exp["in_channels"],
            classes        = 4,
        ).to(device)

        model.load_state_dict(torch.load(exp["checkpoint"], map_location=device))
        print(f"  Loaded: {exp['checkpoint']}")

        metrics = evaluate(model, test_loader, device)

        print(f"  mIoU  = {metrics['miou']:.4f}")
        print(f"  mF1   = {metrics['mf1']:.4f}")
        print(f"  clear = {metrics['iou'][0]:.4f}")
        print(f"  thick = {metrics['iou'][1]:.4f}")
        print(f"  thin  = {metrics['iou'][2]:.4f}")
        print(f"  shadow= {metrics['iou'][3]:.4f}")
        print()

        results[exp["name"]] = metrics

    # save results
    save_path = r"C:\Users\prash\toposegnet\experiments\stage1_test_results.json"
    with open(save_path, "w") as f:
        json.dump(results, f, indent=2)
    print(f"Results saved to: {save_path}")

if __name__ == "__main__":
    main()

Device: cuda

=== 1b — no terrain, augment=True ===


C:\Users\prash\AppData\Local\Temp\ipykernel_10092\68298195.py:110: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(exp["checkpoint"], map_loca

  Loaded: C:\Users\prash\toposegnet\experiments\unet_none_augTrue\best_model.pth


  mIoU  = 0.6859
  mF1   = 0.8030
  clear = 0.8645
  thick = 0.8129
  thin  = 0.4809
  shadow= 0.5855

=== 1a — no terrain, augment=False ===
  Loaded: C:\Users\prash\toposegnet\experiments\unet_none_augFalse\best_model.pth


  mIoU  = 0.7004
  mF1   = 0.8122
  clear = 0.8723
  thick = 0.8361
  thin  = 0.4621
  shadow= 0.6313

Results saved to: C:\Users\prash\toposegnet\experiments\stage1_test_results.json


In [2]:
import sys
sys.path.append(r"C:\Users\prash\toposegnet\src")

import os
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import DataLoader
from tqdm import tqdm
import segmentation_models_pytorch as smp
import json

from dataset import CloudSEN12Dataset

# ── config ───────────────────────────────────────────────────────────────────
# Point this at whichever experiment you want to evaluate on the test set.

CONFIG = {
    "terrain_mode" : "elev",
    "checkpoint"   : r"C:\Users\prash\toposegnet\experiments\unet_elev_augFalse\best_model.pth",
    "batch_size"   : 2,
    "num_workers"  : 0,
    "save_dir"     : r"C:\Users\prash\toposegnet\experiments\unet_elev_augFalse",
}

IN_CHANNELS = {
    "none"   : 13,
    "elev"   : 14,
    "optionA": 15,
    "optionB": 17,
    "optionC": 19,
}

CLASS_WEIGHTS = torch.tensor([1.0000, 2.0384, 5.6464, 6.1658], dtype=torch.float32)


# ── metrics (identical to train.py) ──────────────────────────────────────────
def compute_metrics(preds, labels, num_classes=4):
    """
    preds  : (N, H, W) long tensor — predicted class indices
    labels : (N, H, W) long tensor — ground truth class indices
    returns: dict with per-class IoU, mIoU, per-class F1, mF1
    """
    iou_per_class = []
    f1_per_class  = []

    for c in range(num_classes):
        pred_c  = (preds  == c)
        label_c = (labels == c)

        tp = (pred_c & label_c).sum().item()
        fp = (pred_c & ~label_c).sum().item()
        fn = (~pred_c & label_c).sum().item()

        iou = tp / (tp + fp + fn + 1e-8)
        f1  = 2*tp / (2*tp + fp + fn + 1e-8)

        iou_per_class.append(iou)
        f1_per_class.append(f1)

    return {
        "iou"  : iou_per_class,
        "miou" : float(np.mean(iou_per_class)),
        "f1"   : f1_per_class,
        "mf1"  : float(np.mean(f1_per_class)),
    }


@torch.no_grad()
def test_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_preds  = []
    all_labels = []

    for images, labels in tqdm(loader, desc="  test ", leave=False):
        images = images.to(device)
        labels = labels.to(device).long()

        logits = model(images)
        loss   = criterion(logits, labels)
        total_loss += loss.item()

        preds = logits.argmax(dim=1)
        all_preds.append(preds.cpu())
        all_labels.append(labels.cpu())

    all_preds  = torch.cat(all_preds,  dim=0)
    all_labels = torch.cat(all_labels, dim=0)
    metrics    = compute_metrics(all_preds, all_labels)

    return total_loss / len(loader), metrics


def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device     : {device}")
    print(f"Terrain    : {CONFIG['terrain_mode']}")
    print(f"Checkpoint : {CONFIG['checkpoint']}")

    # ── dataset ───────────────────────────────────────────────────────────
    test_ds = CloudSEN12Dataset(
        split="Test",
        terrain_mode=CONFIG["terrain_mode"],
        augment=False,     # never augment test
    )
    test_loader = DataLoader(
        test_ds,
        batch_size=CONFIG["batch_size"],
        shuffle=False,
        num_workers=CONFIG["num_workers"],
        pin_memory=True,
    )
    print(f"Test patches: {len(test_ds)}")

    # ── model ─────────────────────────────────────────────────────────────
    in_channels = IN_CHANNELS[CONFIG["terrain_mode"]]
    model = smp.Unet(
        encoder_name    = "resnet34",
        encoder_weights = None,     # loading trained weights next, no need for imagenet init
        in_channels     = in_channels,
        classes         = 4,
    ).to(device)

    state_dict = torch.load(CONFIG["checkpoint"], map_location=device)
    model.load_state_dict(state_dict)
    print(f"Model      : U-Net ResNet-34 | in_channels={in_channels} | weights loaded")

    criterion = nn.CrossEntropyLoss(weight=CLASS_WEIGHTS.to(device))

    # ── run ───────────────────────────────────────────────────────────────
    test_loss, metrics = test_epoch(model, test_loader, criterion, device)

    print("\n── Test results ──────────────────────────────")
    print(f"  test_loss = {test_loss:.4f}")
    print(f"  mIoU      = {metrics['miou']:.4f}")
    print(f"  mF1       = {metrics['mf1']:.4f}")
    print(f"  per-class IoU: clear={metrics['iou'][0]:.4f} "
          f"thick={metrics['iou'][1]:.4f} "
          f"thin={metrics['iou'][2]:.4f} "
          f"shadow={metrics['iou'][3]:.4f}")
    print(f"  per-class F1 : clear={metrics['f1'][0]:.4f} "
          f"thick={metrics['f1'][1]:.4f} "
          f"thin={metrics['f1'][2]:.4f} "
          f"shadow={metrics['f1'][3]:.4f}")

    # ── save ──────────────────────────────────────────────────────────────
    result = {
        "terrain_mode": CONFIG["terrain_mode"],
        "checkpoint"  : CONFIG["checkpoint"],
        "test_loss"   : test_loss,
        "miou"        : metrics["miou"],
        "mf1"         : metrics["mf1"],
        "iou_clear"   : metrics["iou"][0],
        "iou_thick"   : metrics["iou"][1],
        "iou_thin"    : metrics["iou"][2],
        "iou_shadow"  : metrics["iou"][3],
        "f1_clear"    : metrics["f1"][0],
        "f1_thick"    : metrics["f1"][1],
        "f1_thin"     : metrics["f1"][2],
        "f1_shadow"   : metrics["f1"][3],
    }

    out_path = os.path.join(CONFIG["save_dir"], "test_results.json")
    with open(out_path, "w") as f:
        json.dump(result, f, indent=2)

    print(f"\nSaved to: {out_path}")


if __name__ == "__main__":
    main()

Device     : cuda
Terrain    : elev
Checkpoint : C:\Users\prash\toposegnet\experiments\unet_elev_augFalse\best_model.pth
Test patches: 975


C:\Users\prash\AppData\Local\Temp\ipykernel_10092\2297172847.py:125: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(CONFIG["checkpoint"], map_location

Model      : U-Net ResNet-34 | in_channels=14 | weights loaded



── Test results ──────────────────────────────
  test_loss = 0.6763
  mIoU      = 0.7009
  mF1       = 0.8127
  per-class IoU: clear=0.8757 thick=0.8317 thin=0.4650 shadow=0.6313
  per-class F1 : clear=0.9337 thick=0.9081 thin=0.6348 shadow=0.7740

Saved to: C:\Users\prash\toposegnet\experiments\unet_elev_augFalse\test_results.json


In [3]:
import sys
sys.path.append(r"C:\Users\prash\toposegnet\src")

import os
import torch
import numpy as np
from torch.utils.data import DataLoader
from tqdm import tqdm
import segmentation_models_pytorch as smp
import json

from dataset import CloudSEN12Dataset

# ── config: point this at the experiment you want to test ───────────────────
CONFIG = {
    "terrain_mode": "optionA",   # must match the trained model's terrain_mode
    "batch_size"  : 2,           # match val batch size used in training
    "num_workers" : 0,           # Windows: must stay 0 (memmap pickling issue)
    "save_dir"    : r"C:\Users\prash\toposegnet\experiments",
}

IN_CHANNELS = {
    "none"   : 13,
    "elev"   : 14,
    "optionA": 15,
    "optionB": 17,
    "optionC": 19,
    "latlon_only": 17,   # Stage 5b(1)
}

CLASS_NAMES = ["clear", "thick", "thin", "shadow"]


# ── metrics (identical logic to train.py's compute_metrics) ─────────────────
def compute_metrics(preds, labels, num_classes=4):
    iou_per_class = []
    f1_per_class  = []

    for c in range(num_classes):
        pred_c  = (preds  == c)
        label_c = (labels == c)

        tp = (pred_c & label_c).sum().item()
        fp = (pred_c & ~label_c).sum().item()
        fn = (~pred_c & label_c).sum().item()

        iou = tp / (tp + fp + fn + 1e-8)
        f1  = 2 * tp / (2 * tp + fp + fn + 1e-8)

        iou_per_class.append(iou)
        f1_per_class.append(f1)

    return {
        "iou" : iou_per_class,
        "miou": float(np.mean(iou_per_class)),
        "f1"  : f1_per_class,
        "mf1" : float(np.mean(f1_per_class)),
    }


@torch.no_grad()
def run_test_eval(model, loader, device):
    model.eval()
    all_preds  = []
    all_labels = []

    for images, labels in tqdm(loader, desc="  test ", leave=False):
        images = images.to(device)
        labels = labels.to(device).long()

        logits = model(images)
        preds  = logits.argmax(dim=1)

        all_preds.append(preds.cpu())
        all_labels.append(labels.cpu())

    all_preds  = torch.cat(all_preds,  dim=0)
    all_labels = torch.cat(all_labels, dim=0)

    return compute_metrics(all_preds, all_labels)


def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device : {device}")

    terrain_mode = CONFIG["terrain_mode"]
    exp_name     = f"unet_{terrain_mode}_augFalse"
    exp_dir      = os.path.join(CONFIG["save_dir"], exp_name)
    ckpt_path    = os.path.join(exp_dir, "best_model.pth")

    print(f"Loading checkpoint: {ckpt_path}")
    if not os.path.exists(ckpt_path):
        raise FileNotFoundError(f"No checkpoint found at {ckpt_path}")

    # ── test dataset ─────────────────────────────────────────────────────────
    test_ds = CloudSEN12Dataset(
        split="Test",
        terrain_mode=terrain_mode,
        augment=False,   # never augment test data
    )
    test_loader = DataLoader(
        test_ds,
        batch_size=CONFIG["batch_size"],
        shuffle=False,
        num_workers=CONFIG["num_workers"],
        pin_memory=True,
    )
    print(f"Test set: {len(test_ds)} patches")

    # ── model ────────────────────────────────────────────────────────────────
    in_channels = IN_CHANNELS[terrain_mode]
    model = smp.Unet(
        encoder_name    = "resnet34",
        encoder_weights = None,   # loading trained weights, no need for imagenet init
        in_channels     = in_channels,
        classes         = 4,
    ).to(device)

    state_dict = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(state_dict)
    print(f"Model loaded: U-Net ResNet-34 | in_channels={in_channels}")

    # ── run eval ─────────────────────────────────────────────────────────────
    metrics = run_test_eval(model, test_loader, device)

    print(f"\n=== Test results: {exp_name} ===")
    print(f"Test mIoU : {metrics['miou']:.4f}")
    print(f"Test mF1  : {metrics['mf1']:.4f}")
    print("Per-class IoU:")
    for name, iou in zip(CLASS_NAMES, metrics["iou"]):
        print(f"  {name:8s}: {iou:.4f}")
    print("Per-class F1:")
    for name, f1 in zip(CLASS_NAMES, metrics["f1"]):
        print(f"  {name:8s}: {f1:.4f}")

    # ── save results ─────────────────────────────────────────────────────────
    results = {
        "experiment"  : exp_name,
        "terrain_mode": terrain_mode,
        "checkpoint"  : ckpt_path,
        "test_miou"   : metrics["miou"],
        "test_mf1"    : metrics["mf1"],
        "test_iou"    : dict(zip(CLASS_NAMES, metrics["iou"])),
        "test_f1"     : dict(zip(CLASS_NAMES, metrics["f1"])),
    }

    out_path = os.path.join(exp_dir, "test_results.json")
    with open(out_path, "w") as f:
        json.dump(results, f, indent=2)
    print(f"\nResults saved to: {out_path}")


if __name__ == "__main__":
    main()

Device : cuda
Loading checkpoint: C:\Users\prash\toposegnet\experiments\unet_optionA_augFalse\best_model.pth
Test set: 975 patches


C:\Users\prash\AppData\Local\Temp\ipykernel_10092\2435223857.py:120: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(ckpt_path, map_location=device)


Model loaded: U-Net ResNet-34 | in_channels=15



=== Test results: unet_optionA_augFalse ===
Test mIoU : 0.6934
Test mF1  : 0.8066
Per-class IoU:
  clear   : 0.8710
  thick   : 0.8307
  thin    : 0.4511
  shadow  : 0.6209
Per-class F1:
  clear   : 0.9310
  thick   : 0.9075
  thin    : 0.6218
  shadow  : 0.7661

Results saved to: C:\Users\prash\toposegnet\experiments\unet_optionA_augFalse\test_results.json


In [1]:
import os
exp_dir = r"C:\Users\prash\toposegnet\experiments\unet_optionA_augFalse"
print(os.listdir(exp_dir))

['best_model.pth', 'config.json', 'history.json', 'test_results.json']


In [7]:

import sys
sys.path.append(r"C:\Users\prash\toposegnet\src")

In [8]:
from evaluate_notebook import run_test_eval

# Stage 3
results_stage3 = run_test_eval(
    terrain_mode="optionA",
    checkpoint_path=r"C:\Users\prash\toposegnet\experiments\unet_optionA_augFalse\best_model.pth"
)

Device       : cuda
Terrain mode : optionA
Checkpoint   : C:\Users\prash\toposegnet\experiments\unet_optionA_augFalse\best_model.pth
Model        : U-Net ResNet-34 | in_channels=15


C:\Users\prash\toposegnet\src\evaluate_notebook.py:56: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(path, map_location=device)


Loaded old-format checkpoint (bare state_dict)



── Test Results ──────────────────────────────────────────────
  mIoU = 0.6934 | mF1 = 0.8066
  per-class IoU: clear=0.8710 thick=0.8307 thin=0.4511 shadow=0.6209
  per-class F1 : clear=0.9310 thick=0.9075 thin=0.6218 shadow=0.7661

Results saved to: C:\Users\prash\toposegnet\experiments\unet_optionA_augFalse\test_results.json


In [9]:
# Stage 4
results_stage4 = run_test_eval(
    terrain_mode="optionB",
    checkpoint_path=r"C:\Users\prash\toposegnet\experiments\unet_optionB_augFalse\best_model.pth"
)

Device       : cuda
Terrain mode : optionB
Checkpoint   : C:\Users\prash\toposegnet\experiments\unet_optionB_augFalse\best_model.pth
Model        : U-Net ResNet-34 | in_channels=17
Loaded new-format checkpoint (epoch=59, best_miou=0.714513357987971)



── Test Results ──────────────────────────────────────────────
  mIoU = 0.6846 | mF1 = 0.8001
  per-class IoU: clear=0.8636 thick=0.8196 thin=0.4386 shadow=0.6168
  per-class F1 : clear=0.9268 thick=0.9009 thin=0.6097 shadow=0.7630

Results saved to: C:\Users\prash\toposegnet\experiments\unet_optionB_augFalse\test_results.json
